In [ ]:
# v2 bundle bootstrap -- locate the bundle root (the first ancestor holding
# both lib/ and data/), put lib/ on sys.path, arm the CPU-f64 evaluation
# convention of the sealed v2 records BEFORE the first jax import, print the
# environment receipt.  `ognrepro.blocked` is imported FIRST: it pins the
# single-thread BLAS env vars at import time, so it must run before numpy
# acquires its thread pool.
import os
import sys
import time

_d = os.path.abspath(os.getcwd())
for _ in range(8):
    if os.path.isdir(os.path.join(_d, "lib")) and os.path.isdir(os.path.join(_d, "data")):
        break
    _d = os.path.dirname(_d)
BUNDLE_ROOT = _d
if os.path.join(BUNDLE_ROOT, "lib") not in sys.path:
    sys.path.insert(0, os.path.join(BUNDLE_ROOT, "lib"))
os.environ.setdefault("OGNREPRO_ROOT", BUNDLE_ROOT)

from ognrepro import blocked as B           # pins single-thread BLAS, then numpy
from ognrepro import repro_common as rc

conv = rc.enable_eval_convention()          # JAX_PLATFORMS=cpu + x64 + matmul 'highest'
assert os.path.samefile(rc.BUNDLE_ROOT, BUNDLE_ROOT), (rc.BUNDLE_ROOT, BUNDLE_ROOT)

import json
import numpy as np

SMOKE = os.environ.get("V2NB_SMOKE", "1") not in ("0", "false", "False", "")
def knob(name, smoke_value, full_value):
    v = os.environ.get(name)
    return int(v) if v else (smoke_value if SMOKE else full_value)

VALUES = os.path.join(BUNDLE_ROOT, "data", "values")
PROV   = os.path.join(BUNDLE_ROOT, "data", "provenance")
CKPT   = os.path.join(BUNDLE_ROOT, "checkpoints")

def load_values(name):
    with open(os.path.join(VALUES, name if name.endswith(".json") else name + ".json")) as fh:
        return json.load(fh)

def cite(doc, label=""):
    """Print the record@sha provenance block of an authored v2 sidecar."""
    prov = doc.get("provenance") or []
    print("  provenance%s: %d sealed record citation(s)" % ((" " + label) if label else "", len(prov)))
    for c in prov[:6]:
        print("    %-42s %-34s %s.." % (c["record"], c["file"], (c["file_sha256"] or "")[:16]))
    if len(prov) > 6:
        print("    ... %d more" % (len(prov) - 6))
    return prov

print("bundle root    :", BUNDLE_ROOT)
print("eval convention:", conv)
print("SMOKE          :", SMOKE, "(V2NB_SMOKE=0 for full size)")
_ = rc.print_env()

# Notebook C (v2) — geometry, comparators and finite-shot behaviour

Every number in this notebook is **replayed from a sealed v2 record**: the
bundle ships the record's own payload (or a faithful transcription with the
record@sha citation attached), and each section re-derives the record's
*decision* from its *inputs* rather than quoting its conclusion.  That is the
non-trivial part: a ratio, a verdict or a band is recomputed here from the
per-lane numbers, and compared with what the record sealed.

Sections: 1. the T0 endpoint pair rule · 2. the geometry battery ·
3. the kernel-ridge comparator · 4. CMP-01 net-over-fit · 5. finite-shot
budgets · 6. cross-host and robustness.

## 1. The T0 endpoint pair rule

In [ ]:
t0p = load_values("v2_t0_pair_values.json")
cite(t0p, "(v2_t0_pair)")
for seed in sorted(t0p["seeds"]):
    s = t0p["seeds"][seed]
    pr = s["pair_rule"]
    lane, ctrl = pr["lane"], pr["control"]
    a, b = s["results"][lane], s["results"][ctrl]
    # re-derive the four ratios from the per-lane numbers
    got = dict(infid_ratio=a["infid"] / b["infid"],
               rho_err_ratio=a["rho_err"] / b["rho_err"],
               stiff_k10_ratio=a["stiff_k10"] / b["stiff_k10"],
               coupling_vs_control=a["coupling_median_err_q"] / b["coupling_median_err_q"],
               coupling_vs_published=a["coupling_median_err_q"] / pr["published_median"])
    bad = {k: (got[k], pr["ratios"][k]) for k in got
           if abs(got[k] - pr["ratios"][k]) > 1e-9 * max(1.0, abs(pr["ratios"][k]))}
    thr = pr["thresholds"]
    checks = dict(i_infid=got["infid_ratio"] <= thr["infid_ratio"],
                  ii_rho=got["rho_err_ratio"] <= thr["rho_err_ratio"],
                  iii_stiff=got["stiff_k10_ratio"] <= thr["stiff_k10_ratio"],
                  guard_control=got["coupling_vs_control"] <= thr["coupling_vs_control"],
                  guard_published=got["coupling_vs_published"] <= thr["coupling_vs_published"])
    verdict = "GAIN TRANSFERRED" if all(checks.values()) else "NOT TRANSFERRED"
    print("%-6s %-28s vs %-24s n=%d" % (seed, lane, ctrl, s["n"]))
    for k in sorted(got):
        print("      %-24s %.6f  (sealed %.6f, threshold %.3f)"
              % (k, got[k], pr["ratios"][k], thr.get(k, float("nan"))))
    print("      re-derived verdict %-18s sealed %-18s %s"
          % (verdict, pr["verdict"], "MATCH" if verdict == pr["verdict"] else "MISMATCH"))
    assert not bad, bad
    assert checks == pr["checks"], (checks, pr["checks"])
    assert verdict == pr["verdict"]
print("\n[T0] every T0-PAIR ratio, check and verdict re-derived from the sealed per-lane numbers  PASS")

## 2. The geometry battery

In [ ]:
geo = load_values("v2_geo_battery_values.json")
cite(geo, "(v2_geo_battery)")
print("record:", geo["record"], "supersedes:", geo.get("supersedes"))
res = geo["result"]
print("schema:", res.get("schema"), " all gates pass:", res.get("all_gates_pass"))
gates = res.get("gates") or {}
for k in sorted(gates):
    print("   %-44s %s" % (k, gates[k]))
assert res.get("all_gates_pass") is not False
rr = res.get("results") or res.get("per_model") or {}
if isinstance(rr, dict) and rr:
    keys = sorted(rr)
    print("\narms:", len(keys))
    k0 = rr[keys[0]]
    fields = [f for f in sorted(k0) if isinstance(k0[f], (int, float))][:8]
    print("%-30s %s" % ("arm", "  ".join("%14s" % f for f in fields)))
    for k in keys:
        print("%-30s %s" % (k, "  ".join("%14.6e" % rr[k][f] if isinstance(rr[k].get(f), (int, float))
                                         else "%14s" % "-" for f in fields)))

## 3. The kernel-ridge comparator

In [ ]:
krr = load_values("v2_krr_values.json")
cite(krr, "(v2_krr)")
kr = krr["result"]
print("schema:", kr.get("schema"), " stage:", kr.get("stage"))
for k in sorted(kr):
    v = kr[k]
    if isinstance(v, (int, float, str, bool)) or v is None:
        print("   %-32s %s" % (k, v))
clean = load_values("v2_clean_medians.json")
best = min((v["median_err_q"], l) for l, v in clean["lanes"].items()
           if v.get("state_type") == "thermal")
print("\nbest sealed v2 thermal OGN clean median: %.6e  (%s)" % best)

## 4. CMP-01 — the network against the registered fit, per noise budget

In [ ]:
cmp01 = load_values("v2_cmp01_values.json")
cite(cmp01, "(v2_cmp01)")
cells = cmp01["cells"]
print("CMP-01 cells:", len(cells))
one = cells[sorted(cells)[0]]
print("convention:", json.dumps(one.get("convention"), indent=1)[:600])
cols = one.get("columns")
rows = one.get("index_rows")
if cols and rows:
    print("\ncolumns:", cols if isinstance(cols, list) else list(cols)[:10])
    print("rows   :", len(rows))
    for r in rows[:8]:
        print("   ", r if not isinstance(r, dict) else {k: r[k] for k in list(r)[:6]})

## 5. Finite-shot budgets

In [ ]:
shots = load_values("v2_shots_nodes6_values.json")
cite(shots, "(v2_shots_nodes6)")
lanes = sorted(shots["lanes"])
print("lanes with a sealed 6-node sweep:", len(lanes))
ref = shots["lanes"][lanes[0]]
print("budget nodes:", ref["nodes"])
print("N_s         :", ["%.3g" % v for v in ref["N_s"]])
print("noise models:", ref["models"], " states:", ref["n_states"], " draws:", ref["n_noise"])
print()
for lane in lanes:
    v = shots["lanes"][lane]
    assert v["nodes"] == ref["nodes"] and v["N_s"] == ref["N_s"], lane
    r = v["results"]
    if isinstance(r, dict):
        med = None
        for key in ("median", "medians", "median_err", "per_node"):
            if key in r:
                med = r[key]
                break
        print("  %-34s gates %s  keys %s" % (lane, v.get("all_gates_pass"), sorted(r)[:6]))
print("\n[T0] every sealed sweep shares the same 6 budget nodes and N_s ladder  PASS")

## 6. Cross-host reproducibility and the robustness battery

In [ ]:
for name, label in (("v2_xfleet_values.json", "cross-host (V2-XFLEET-01)"),
                    ("v2_crosseval_values.json", "cross-evaluation"),
                    ("v2_robustness_values.json", "robustness battery"),
                    ("v2_beta1_panel_values.json", "beta=1 panel")):
    p = os.path.join(VALUES, name)
    if not os.path.isfile(p):
        print("%-28s (not in this dist)" % label)
        continue
    doc = load_values(name)
    print("== %s -- %s" % (label, name))
    cite(doc)
    res = doc.get("result") or doc.get("seeds") or {}
    if isinstance(res, dict):
        flat = {k: v for k, v in res.items() if isinstance(v, (int, float, str, bool))}
        for k in sorted(flat)[:10]:
            print("     %-34s %s" % (k, flat[k]))
        sub = {k: v for k, v in res.items() if isinstance(v, dict)}
        for k in sorted(sub)[:4]:
            print("     %-34s {%s}" % (k, ", ".join(sorted(sub[k])[:6])))
    print()
print("notebook C complete")